In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Load the persisted data from Notebook 1
df_clean = pd.read_parquet('group2_stocks.parquet')
df_benchmark = pd.read_parquet('benchmark.parquet')
# 2. Re-calculate Log Returns (The "Price Profile")
# Mathematically: $r_t = \ln(P_t / P_{t-1})$
log_returns = np.log(df_clean / df_clean.shift(1)).fillna(0)

print(f"✅ Data loaded successfully. stocks: {df_clean.shape}, benchmark: {df_benchmark.shape}")

### Traditional K Means Clustering

In [ ]:
# --- CELL 1: PRICE PROFILE CLUSTERING (K-MEANS) ---


# 1. FEATURE PREPARATION
# Transpose so rows = Stocks, columns = Daily Returns
X = log_returns.T 
X = X.fillna(0) 

# Standardization (Z-Score)
# Required by Guardrails to ensure high-price stocks don't bias the distance metrics.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. DIMENSIONALITY REDUCTION (PCA)
# We fit the PCA first, then we can print its results.
pca = PCA(n_components=0.90) # Keep components explaining 90% of variance
X_pca = pca.fit_transform(X_scaled)

# FIXED: Moved these prints to AFTER the pca.fit_transform call
print(f"Feature Matrix Shape: {X.shape}")
print(f"PCA Reduced Shape: {X_pca.shape} (Noise reduced)")
print("Number of components selected:", pca.n_components_)
print("Explained variance ratio total:", pca.explained_variance_ratio_.sum())

# 3. OPTIMAL K SELECTION (Elbow & Silhouette)
inertias = []
silhouettes = []
K_range = range(2, 12) 

print("\nRunning K-Means for different k values...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca)
    
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels))
    print(f"k={k}: Silhouette Score = {silhouettes[-1]:.4f}")

# 4. VISUALIZATION
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(K_range, inertias, 'bo-')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (Distortion)')
ax1.set_title('Elbow Method: Optimal k')
ax1.grid(True)

ax2.plot(K_range, silhouettes, 'rs-')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Analysis (Higher is Better)')
ax2.grid(True)
plt.tight_layout()
plt.show()

# 5. FINAL CLUSTERING
# Automatically pick the k with the highest Silhouette score
optimal_k = list(K_range)[np.argmax(silhouettes)]
print(f"\nProceeding with Optimal k = {optimal_k}")

final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = final_kmeans.fit_predict(X_pca)

# Store results in a DataFrame (results_df) for use in subsequent cells
results_df = pd.DataFrame({
    'Ticker': X.index,
    'Cluster': clusters
}).sort_values('Cluster')

print("\n--- Cluster Assignments (Head) ---")
print(results_df.head(10))

In [ ]:
# --- REGENERATING THE 'MERGED' DATAFRAME (FIXED) ---

# 1. Calculate Annualized Returns and Volatility
# log_returns shape: (Days, Stocks)
mean_ann = log_returns.mean() * 252
vol_ann = log_returns.std() * np.sqrt(252)
sharpe = mean_ann / vol_ann

# 2. FIXED: Calculate Beta 
# We use .squeeze() to turn the benchmark DataFrame into a Series
# This resolves the "size 500 vs size 1" ValueError
log_ret_bench = np.log(df_benchmark / df_benchmark.shift(1)).fillna(0).squeeze()
var_bench = log_ret_bench.var()

betas = {}
for ticker in log_returns.columns:
    # Now that log_ret_bench is a Series, .cov() will work perfectly
    covariance = log_returns[ticker].cov(log_ret_bench)
    betas[ticker] = covariance / var_bench

# 3. Combine into a Metrics DataFrame
# We ensure the index matches the active tickers
metrics_df = pd.DataFrame({
    'Ticker': log_returns.columns,
    'Mean_Ann': mean_ann.values,
    'Vol_Ann': vol_ann.values,
    'Sharpe': sharpe.values,
    'Beta': [betas[t] for t in log_returns.columns] # Ensure alignment
})

# 4. Merge with your Cluster Assignments (results_df)
# results_df must exist from your K-Means cell
merged = pd.merge(results_df, metrics_df, on='Ticker')

print(f"✅ 'merged' DataFrame reconstructed for {len(merged)} stocks.")


### Cluster-Level Aggregation & Performance Analysis

In [ ]:
# -----------------------------
# CELL: Cluster-level summaries
# -----------------------------
cluster_summary = merged.groupby('Cluster').agg(
    Count = ('Ticker','count'),
    Mean_MeanAnn = ('Mean_Ann','mean'),
    Mean_VolAnn  = ('Vol_Ann','mean'),
    Mean_Sharpe  = ('Sharpe','mean'),
    Mean_Beta    = ('Beta','mean')
).round(4)

display(cluster_summary)


### Visualizing the clusters

In [ ]:
# -----------------------------
# CELL: Plot Risk-Return scatter (Plotly)
# -----------------------------
import plotly.express as px

fig = px.scatter(
    merged, x='Vol_Ann', y='Mean_Ann', color='Cluster',
    hover_data=['Ticker','Beta','Sharpe'],
    title='Annualized Risk-Return by Cluster',
    labels={'Vol_Ann':'Annualized Volatility','Mean_Ann':'Annualized Mean Return'}
)
fig.update_layout(height=600, width=800)
fig.show()


In [ ]:
results_df.to_csv('labels_kmeans.csv', index=False)